# Chapter 3
Study notes for *Active Inference* (Parr, Pezzulo & Friston 2022), Chapter 3: **Markov blankets, surprise, and variational free energy**.

## Markov Blankets

A **Markov blanket** is the set of variables that mediate the statistical influence between a system and its environment. In a generative model, the blanket of a node renders that node **conditionally independent** of everything outside the blanket, given the blanket itself.

From page 43: *"... a Markov blanket is the set of variables that mediate all (statistical) interactions between a system and its environment."*

- In a **Bayesian network** (directed), the Markov blanket of a node is its **parents**, its **children**, and its children's **other parents** (the *co-parents*).
- In a **Markov network** (undirected), the Markov blanket of a node is simply its **neighbours**.

The blanket partitions a sentient system into **internal states**, **external states**, and the **blanket states** (sensory and active) that separate them. Particles that possess a Markov blanket can be said to exist as identifiable, self-evidencing systems.

#### Resources

*Bayesian Networks in Python* — https://github.com/ncullen93/pyBN

*What does it mean to calculate a Markov Blanket of a node?* — https://stackoverflow.com/questions/49038111/how-to-find-markov-blanket-for-a-node

-> How does a Markov Blanket of a node relate to Active Inference? (Every self-organising system is defined by one.)

*Markov Network library* — https://pgmpy.org/models/markovnetwork.html#pgmpy.models.MarkovNetwork.MarkovNetwork.markov_blanket

A Markov blanket is the set of neighbouring nodes of the given node (Markov network treatment).

In [ ]:
# Markov blanket of a node in a Bayesian network: parents + children + co-parents.
def markov_blanket_bn(adj, node):
    """adj: dict node -> list of child nodes (directed graph)."""
    parents  = [u for u, kids in adj.items() if node in kids]
    children = list(adj.get(node, []))
    coparents = [u for v in children for u, ks in adj.items() if u != node and v in ks]
    return sorted(set(parents + children + coparents))

# Small Bayesian network: A -> B, B -> D, C -> B
adj = {'A': ['B'], 'B': ['D'], 'C': ['B'], 'D': []}
blanket = markov_blanket_bn(adj, 'B')
print('Markov blanket of B:', blanket)
assert blanket == ['A', 'C', 'D']

## Surprise Minimization and Hamilton's Principle of Least Action

An adaptive system acts to keep itself in a set of **surprising** states rare. The **surprise** (self-information) of an observation is

    surprise = -ln P(y)

Minimising surprise over time is equivalent to maximising **model evidence** P(y) — the system is *self-evidencing*. Active Inference casts this as a path-integral problem: over time the system's trajectory minimises its free energy, in direct analogy to **Hamilton's principle of least action**, where classical trajectories extremise the action.

In [ ]:
import math

# Surprise (self-information) of an observation y: -ln P(y).
# Less probable events carry more surprise; minimising surprise is
# equivalent to maximising model evidence P(y).
for p in (0.9, 0.5, 0.1, 0.01):
    print(f'P(y)={p:<5} surprise = -ln P(y) = {-math.log(p):.3f} nats')

## Variational Free Energy, Model Evidence, and Surprise

The **variational free energy** provides an upper bound on surprise that is tractable to optimise:

    F(q) = KL[Q(x) || P(x|y)] - ln P(y)

Because the Kullback-Leibler divergence is never negative, F(q) >= -ln P(y). The bound is tight when the variational posterior Q(x) equals the true posterior P(x|y), so **minimising F drives Q towards the exact posterior** — perception as inference. See **Chapters/VFE.jl** (a Pluto notebook) for an interactive simulator of this surface.

In [ ]:
import math

# Variational free energy as a function of variational-posterior mass q = Q(x=1):
#   F(q) = KL[Q(x) || P(x|y)] - ln P(y)   (KL part shown here)
def xlogy(x, y):
    return 0.0 if x <= 0 else x * math.log(x / y)

def kl(q, p):
    return xlogy(q, p) + xlogy(1 - q, 1 - p)

p = 0.7   # exact posterior mass P(x=1|y)
qstar = min((q / 100 for q in range(1, 100)), key=lambda q: kl(q, p))
print(f'minimiser q* ~ {qstar:.2f}   (exact posterior p = {p})')
assert abs(qstar - p) < 0.02

## Running the code

All Python cells above use only the standard library (no third-party dependencies), so they run on a stock Python 3 kernel. The interactive surface for variational free energy lives in **Chapters/VFE.jl** (open with Pluto.jl).